In [1]:
import os
import re
import csv
import numpy as np
import pandas as pd
import mendeleev as md
from math import gcd
from ase.io import read
import matplotlib.pyplot as plt

# Show float values with 2 decimal places when displaying DataFrames.
pd.options.display.float_format = '{:.2f}'.format

# Ensure wide DataFrames print all columns (no truncation).
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 20000)
pd.set_option('display.expand_frame_repr', True)

root = '/Users/jiuy97/Desktop/3_RuO2/7_prediction'
if not os.path.exists(root):
    root = '/Users/hailey/Desktop/3_RuO2/7_prediction'
# root = '/Users/hailey/Desktop/3_RuO2/6_ICOHP/4_slab_M-RuO2'
# oxygen_potential = -4.658724749999999 # 300K
oxygen_potential = -4.658724749999999+0.27-0.73 # 700K
# oxygen_potential = -4.658724749999999+0.27-0.85 # 800K
figsize = (8, 6)
dpi = 300

In [2]:
element_indices = list(range(21, 31)) + list(range(39, 49)) + [57] + list(range(72, 81))
elements = [md.element(i).symbol for i in element_indices]
data = pd.DataFrame(index=elements)
data['a'] = element_indices

In [ ]:
# os: oxidation state
for i in element_indices:
    element = md.element(i).symbol
    for dir in ['1_top_cus_M', '2_top_brg_M', '3_sub_cus_M', '4_sub_brg_M', '5_top_cus_Ru', '6_top_brg_Ru', '7_sub_cus_Ru', '8_sub_brg_Ru']:
        doping_site = dir.split('_')[1] + '_' + dir.split('_')[2]
        doping_metal = dir.split('_')[3]
        path = os.path.join(root, '3_surface_Ru', dir, f'{i}_{element}')
        energy_file = os.path.join(path, 'final_with_calculator.json')
        # moment_file = os.path.join(path, 'moments.json')
        # charge_file = os.path.join(path, 'atoms_bader_charge.json')
        row_updates = {}
        if os.path.exists(energy_file):
            atoms = read(energy_file)
            # metal_indices = [i for i, atom in enumerate(atoms) if atom.symbol in elements]
            # n_atoms = len(atoms)
            # n_oxygens = sum(atom.symbol == 'O' for atom in atoms)
            # n_metals = sum(atom.symbol != 'O' for atom in atoms)
            # oxidation_state = 2 * n_oxygens / n_metals
            energy = atoms.get_potential_energy()
            row_updates[f'{doping_site}_{doping_metal}_e'] = energy
        # if os.path.exists(moment_file):
        #     atoms = read(moment_file)
        #     moments = atoms.get_magnetic_moments()
        #     if oxide_type == 'M-RuO2':
        #         row_updates['M-RuO2_m'] = np.mean(np.abs(moments[0:7]))
        #         for j, moment in enumerate(moments[metal_indices]):
        #             row_updates[f'M-RuO2_m{j}'] = moment
        #     elif oxide_type == 'M-IrO2':
        #         row_updates['M-IrO2_m'] = np.mean(np.abs(moments[0:7]))
        #         for j, moment in enumerate(moments[metal_indices]):
        #             row_updates[f'M-IrO2_m{j}'] = moment
        #     else:
        #         row_updates[f'{oxide_type}_m'] = np.mean(np.abs(moments[metal_indices]))
        # if os.path.exists(charge_file):
        #     atoms = read(charge_file)
        #     charges = atoms.get_initial_charges()
        #     if oxide_type == 'M-RuO2':
        #         row_updates['M-RuO2_c'] = np.mean(np.abs(charges[0:7]))
        #         for j, charge in enumerate(charges):
        #             row_updates[f'M-RuO2_c{j}'] = charge
        #     elif oxide_type == 'M-IrO2':
        #         row_updates['M-IrO2_c'] = np.mean(np.abs(charges[0:7]))
        #         for j, charge in enumerate(charges):
        #             row_updates[f'M-IrO2_c{j}'] = charge
        #     else:
        #         row_updates[f'{oxide_type}_c'] = np.mean(np.abs(charges[metal_indices]))
        if row_updates:
            data.loc[element, list(row_updates)] = list(row_updates.values())

# de: doping energy
# fe: formation energy
data['RuO2_top_cus_de'] = data['top_cus_M_e'] - 7/8*data['MO2_e']['Ru'] - 1/8*data['MO2_e']
# data['M-RuO2_fe1'] = data['M-RuO2_e'] - 7/8*data['M_e']['Ru'] - 1/8*data['M_e'] - 2*oxygen_potential
# data['M-RuO2_fe2'] = data['M-RuO2_e'] - 7/8*data['MO2_e']['Ru'] - 1/8*data['MO2_e']
# data['M-RuO2_fe3'] = data['M-RuO2_e'] - 7/8*data['MxOy_e']['Ru'] - 1/8*data['MxOy_e'] - (2-data['MxOy_y']/data['MxOy_x'])/8*oxygen_potential
# data['M-IrO2_fe1'] = data['M-IrO2_e'] - 7/8*data['M_e']['Ir'] - 1/8*data['M_e'] - 2*oxygen_potential
# data['M-IrO2_fe2'] = data['M-IrO2_e'] - 7/8*data['MO2_e']['Ir'] - 1/8*data['MO2_e']
# data['M-IrO2_fe3'] = data['M-IrO2_e'] - 7/8*data['MxOy_e']['Ir'] - 1/8*data['MxOy_e'] - (2-data['MxOy_y']/data['MxOy_x'])/8*oxygen_potential
# # data['MxOy_fe'] = data['MxOy_e'] - data['MO2_e'] - (2-data['MxOy_y']/data['MxOy_x'])*oxygen_potential
# data.drop(columns=['MO2_e', 'MxOy_e', 'M-RuO2_e', 'M-IrO2_e'], inplace=True)
# data.loc['Nb', 'MxOy_x'] = 2
# data.loc['Nb', 'MxOy_y'] = 5
data


,a,top_cus_M_e,top_brg_M_e,top_cus_Ru_e,top_brg_Ru_e
Sc,21,-408.40,-407.23,NaN,NaN
Ti,22,-403.70,-403.53,NaN,NaN
V,23,-400.58,-399.23,NaN,NaN
Cr,24,-401.22,-399.54,NaN,NaN
Mn,25,-398.52,-396.53,NaN,NaN
Fe,26,-392.91,-391.03,NaN,NaN
Co,27,-387.15,-384.96,NaN,NaN
Ni,28,-380.77,NaN,NaN,NaN
Cu,29,-372.46,NaN,NaN,NaN
Zn,30,-376.26,NaN,NaN,NaN
